# 02 – Preprocessing & NLP Feature Extraction (PhreshPhish)

**Goals**
- Load data via Hugging Face `datasets` (streaming)
- Clean & truncate HTML safely (never keep full HTML in memory)
- Extract visible text + useful structural signals
- Create URL lexical features
- Prepare clean text for classical ML and Transformer models
- No synthetic / sample data creation

In [1]:
import warnings
warnings.filterwarnings("ignore")

from datasets import load_dataset
import pandas as pd
import numpy as np
import re
from urllib.parse import urlparse
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
import joblib
from pathlib import Path

tqdm.pandas()

In [2]:
## 1. Configuration

DATASET_NAME = "phreshphish/phreshphish"
N_SAMPLES = 8_000          # adjust according to your RAM (start with 5k–10k)
MAX_HTML_CHARS = 30_000    # hard truncate before BeautifulSoup
MAX_TEXT_CHARS = 4_000     # final text fed to models

OUTPUT_DIR = Path("processed")
OUTPUT_DIR.mkdir(exist_ok=True)

In [3]:
## 2. Helper Functions

def clean_html(html: str, max_chars: int = MAX_HTML_CHARS) -> str:
    """Aggressive but safe HTML cleaning + truncation."""
    if not isinstance(html, str) or len(html) == 0:
        return ""
    
    # Hard truncate early to protect memory
    html = html[:max_chars]
    
    try:
        soup = BeautifulSoup(html, "lxml")   # lxml is faster & more robust
    except Exception:
        return ""
    
    # Remove noisy tags
    for tag in soup(["script", "style", "noscript", "svg", "iframe", "canvas"]):
        tag.decompose()
    
    # Get visible text
    text = soup.get_text(separator=" ", strip=True)
    
    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text[:MAX_TEXT_CHARS]


def extract_url_features(url: str) -> dict:
    """Lightweight lexical URL features."""
    try:
        parsed = urlparse(str(url))
        netloc = parsed.netloc.lower()
        path = parsed.path.lower()
        full = str(url).lower()
        
        return {
            "url_len": len(full),
            "domain_len": len(netloc),
            "path_len": len(path),
            "num_dots": full.count("."),
            "num_hyphens": full.count("-"),
            "num_underscores": full.count("_"),
            "num_digits": sum(c.isdigit() for c in full),
            "num_params": full.count("="),
            "has_https": int(full.startswith("https")),
            "has_ip": int(bool(re.search(r"\d+\.\d+\.\d+\.\d+", netloc))),
            "num_subdomains": max(0, netloc.count(".") - 1),
        }
    except Exception:
        return {k: 0 for k in [
            "url_len", "domain_len", "path_len", "num_dots", "num_hyphens",
            "num_underscores", "num_digits", "num_params", "has_https",
            "has_ip", "num_subdomains"
        ]}


def process_example(example):
    """Process a single HF example → clean dict."""
    html_raw = example.get("html") or ""
    url = example.get("url") or ""
    
    clean_text = clean_html(html_raw)
    url_feats = extract_url_features(url)
    
    return {
        "url": url,
        "label": example.get("label"),
        "target": example.get("target"),
        "lang": example.get("lang"),
        "text": clean_text,
        "text_len": len(clean_text),
        "html_len_raw": len(html_raw),
        **url_feats
    }

In [5]:
## 3. Stream & Process Data (Memory Safe)

print("Loading streaming dataset...")
ds = load_dataset(
    DATASET_NAME,
    split="train",
    streaming=True,
    trust_remote_code=False
)

print(f"Processing {N_SAMPLES} examples...")
processed = []

for i, example in enumerate(tqdm(ds, total=N_SAMPLES)):
    if i >= N_SAMPLES:
        break
    processed.append(process_example(example))

df = pd.DataFrame(processed)
print(f"Processed shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df.head(2)

Loading streaming dataset...


Resolving data files:   0%|          | 0/56 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/56 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/21 [00:00<?, ?it/s]

Processing 8000 examples...


  0%|          | 0/8000 [00:00<?, ?it/s]

'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: 785c8df4-58c2-4d10-89cb-3a884d944fd0)')' thrown while requesting GET https://huggingface.co/datasets/phreshphish/phreshphish/resolve/eabec4b7a66324b79cc8a0ad856d1731dc26fe1a/data/train-002.parquet
Retrying in 1s [Retry 1/5].


Processed shape: (8000, 18)
Memory usage: 10.1 MB


,url,label,target,lang,text,text_len,html_len_raw,url_len,domain_len,path_len,num_dots,num_hyphens,num_underscores,num_digits,num_params,has_https,has_ip,num_subdomains
0,https://www.lifeonthemediterranean.com/portofi...,benign,None,en,Is Portofino Italy worth a visit? - Life On Th...,61,518982,69,30,31,2,4,0,0,0,1,0,1
1,https://moneypuck.com/goalies.htm,benign,None,en,NHL Goalie Advanced Stats 2024-2025 Games Play...,1861,192077,33,13,12,2,0,0,0,0,1,0,0


In [7]:
## 4. Basic Cleaning & Filtering

print("Before filtering:", len(df))

# Remove completely empty text
df = df[df["text_len"] > 20].copy()

# Optional: keep only English for first experiments
# df = df[df["lang"] == "en"].copy()

print("After filtering:", len(df))
print("\nLabel distribution:")
print(df["label"].value_counts(normalize=True).round(3) * 100)

Before filtering: 6415
After filtering: 6415

Label distribution:
label
benign    56.1
phish     43.9
Name: proportion, dtype: float64


In [8]:
## 5. Create final text field for models

## We concatenate URL + cleaned HTML text.  
## This works well for both TF-IDF and Transformer models.

df["input_text"] = df["url"].fillna("") + " [SEP] " + df["text"].fillna("")
df["input_text"] = df["input_text"].str.slice(0, 4500)   # safety

print("Example input_text:")
print(df["input_text"].iloc[0][:400], "...")

Example input_text:
https://www.lifeonthemediterranean.com/portofino-italy-worth-a-visit/ [SEP] Is Portofino Italy worth a visit? - Life On The Mediterranean ...


In [9]:
## 6. Encode labels

label2id = {"benign": 0, "phish": 1}
id2label = {0: "benign", 1: "phish"}

df["label_id"] = df["label"].map(label2id)

print(df["label_id"].value_counts())

label_id
0    3602
1    2813
Name: count, dtype: int64


In [12]:
## 7. Save processed data

# We save a lightweight parquet (no raw HTML).

save_path = OUTPUT_DIR / "phreshphish_processed.parquet"
df.to_parquet(save_path, index=False)
print(f"Saved → {save_path}")
print(f"File size: {save_path.stat().st_size / 1e6:.1f} MB")

Saved → processed\phreshphish_processed.parquet
File size: 6.2 MB


In [11]:
## 8. Quick feature overview

feature_cols = [
    "url_len", "domain_len", "path_len", "num_dots", "num_hyphens",
    "num_underscores", "num_digits", "num_params", "has_https",
    "has_ip", "num_subdomains", "text_len", "html_len_raw"
]

print(df.groupby("label")[feature_cols].mean().round(1).T)


label              benign    phish
url_len              61.5     55.8
domain_len           17.4     25.0
path_len             36.2     16.3
num_dots              2.1      2.1
num_hyphens           2.5      1.3
num_underscores       0.3      0.2
num_digits            2.8      5.6
num_params            0.0      0.2
has_https             1.0      0.9
has_ip                0.0      0.0
num_subdomains        0.9      0.9
text_len            546.7    714.2
html_len_raw     371069.4  93020.0


## 9. Summary

- Streaming load + aggressive truncation → no MemoryError
- Clean visible text extracted with BeautifulSoup
- URL lexical features ready for classical ML
- `input_text` ready for TF-IDF or Transformer models
- Lightweight parquet saved for next notebooks

**Next** → `03_ml_baselines.ipynb`